# pycograd — visualizing the graph IR

pycograd's **graph mode** records the primitives a `|>` pipeline runs into a flat
SSA graph (`capture`), differentiates it (`value_and_grad` / `vjp_graph`), and optimizes
it (`optimize`). This notebook shows how to **see** those graphs:

* a bare `Graph` in a cell **auto-renders** (an SVG picture when `graphviz` is
  installed, otherwise a jaxpr-style text listing),
* `graph.pretty()` is the text listing, `graph.to_dot()` is Graphviz DOT source.

> For the SVG pictures you need the `graphviz` **Python package** *and* the Graphviz
> `dot` **binary** on your PATH (`pip install graphviz`, plus e.g. `brew install
> graphviz` / `apt-get install graphviz`). Without them, displaying a graph falls
> back to the text listing — everything below still works.

In [1]:
%load_ext pycograd

## 1. Capture a `|>` pipeline into the graph IR

Parameters go in a `params{...} as weights:` block; the forward is a `|>` pipeline.
`capture(forward, x)` runs it once with a tracer input and records every op into a
`Graph`. Displaying the bare `g` auto-renders.

> `$` is the piped value for a stage. Reusing the *same* value within a stage needs a
> **named** placeholder — `$h * $h` squares `h`; a bare `$ * $` would be two holes.

In [2]:
import numpy as np
from pycograd import capture, value_and_grad, optimize, d_sigmoid, ops
from pycograd.transpose import vjp_graph

rng = np.random.default_rng(0)
x = rng.standard_normal((4, 3))

with params{
    w = rng.standard_normal((3, 2))
    b = np.zeros(2)
} as weights:
    # sum(tanh(x @ w + b) ** 2) as a pipeline; $h * $h reuses the piped value.
    forward = $ |> $ @ w + b |> np.tanh |> $h * $h |> np.sum
    g = capture(forward, x)

g            # bare Graph -> auto-renders (SVG if graphviz is installed, else the listing)

graph(%0:f64[4,3]) {
  %1 = weight 'w' -> f64[3,2]
  %2 = matmul %0 %1 -> f64[4,2]
  %3 = weight 'b' -> f64[2]
  %4 = add %2 %3 -> f64[4,2]
  %5 = tanh %4 -> f64[4,2]
  %6 = mul %5 %5 -> f64[4,2]
  %7 = sum %6 -> f64[]
  outputs: %7
}

`repr` stays a terse one-liner (so logging isn't noisy); `print` / `.pretty()` give
the full SSA listing — `%id = prim args {params} -> dtype[shape]`.

In [3]:
print(repr(g))
print()
print(g.pretty())

Graph(1 in, 5 ops, 1 out)

graph(%0:f64[4,3]) {
  %1 = weight 'w' -> f64[3,2]
  %2 = matmul %0 %1 -> f64[4,2]
  %3 = weight 'b' -> f64[2]
  %4 = add %2 %3 -> f64[4,2]
  %5 = tanh %4 -> f64[4,2]
  %6 = mul %5 %5 -> f64[4,2]
  %7 = sum %6 -> f64[]
  outputs: %7
}


### The DOT source
`graph.to_dot()` returns Graphviz DOT text you can render anywhere (e.g. write to a
file and run `dot -Tpng g.dot -o g.png`). `graph.render()` returns a
`graphviz.Source` for inline display when the package is installed.

In [4]:
print(g.to_dot())

digraph G {
  rankdir=TB;
  node [shape=box, fontname="monospace"];
  0 [label="%0 in\nf64[4,3]", shape=ellipse, style=filled, fillcolor=lightblue];
  1 [label="%1 weight w\nf64[3,2]", shape=ellipse, style=filled, fillcolor=lightgreen];
  2 [label="%2 matmul\nf64[4,2]"];
  0 -> 2;
  1 -> 2;
  3 [label="%3 weight b\nf64[2]", shape=ellipse, style=filled, fillcolor=lightgreen];
  4 [label="%4 add\nf64[4,2]"];
  2 -> 4;
  3 -> 4;
  5 [label="%5 tanh\nf64[4,2]"];
  4 -> 5;
  6 [label="%6 mul\nf64[4,2]"];
  5 -> 6;
  7 [label="%7 sum\nf64[]", peripheries=2];
  6 -> 7;
}


## 2. Forward **and** backward in one graph: `value_and_grad`

`value_and_grad` differentiates the captured forward into a single graph computing
`(value, grads)` — forward and backward together. That's what lets the optimizer
work *across* the forward/backward boundary.

In [5]:
gg = value_and_grad(g)   # outputs: value, then one grad per input leaf
gg

graph(%0:f64[4,3]) {
  %1 = weight 'w' -> f64[3,2]
  %2 = matmul %0 %1 -> f64[4,2]
  %3 = weight 'b' -> f64[2]
  %4 = add %2 %3 -> f64[4,2]
  %5 = tanh %4 -> f64[4,2]
  %6 = mul %5 %5 -> f64[4,2]
  %7 = sum %6 -> f64[]
  %8 = const 1.0 -> f64[]
  %9 = broadcast_to %8 [4, 2] -> f64[4,2]
  %10 = mul %9 %5 -> f64[4,2]
  %11 = mul %9 %5 -> f64[4,2]
  %12 = add %10 %11 -> f64[4,2]
  %13 = tanh %4 -> f64[4,2]
  %14 = tanh %4 -> f64[4,2]
  %15 = mul %13 %14 -> f64[4,2]
  %16 = sub 1.0 %15 -> f64[4,2]
  %17 = mul %12 %16 -> f64[4,2]
  %18 = sum %17 {axis=0} -> f64[2]
  %19 = transpose %1 [1, 0] -> f64[2,3]
  %20 = matmul %17 %19 -> f64[4,3]
  %21 = transpose %0 [1, 0] -> f64[3,4]
  %22 = matmul %21 %17 -> f64[3,2]
  outputs: %7, %18, %22
}

## 3. Cross-pass optimization you can *see*

`d_sigmoid`'s gradient is `g · σ(x) · (1 − σ(x))` — it **recomputes** `σ(x)`. So the
combined forward+backward graph contains the forward's `σ(x)` plus the two in the
backward. `optimize` runs CSE **across the boundary** and merges them into one.

In [6]:
sig = $ |> d_sigmoid |> np.sum          # the VJP recomputes sigmoid(x)
xs = rng.standard_normal((3, 4))
combined = value_and_grad(capture(sig, xs))
opt = optimize(combined)

n_before = sum(1 for n in combined.nodes if n.prim is ops.d_sigmoid)
n_after  = sum(1 for n in opt.nodes      if n.prim is ops.d_sigmoid)
print(f"d_sigmoid nodes  before optimize: {n_before}   after: {n_after}")
opt   # the merged graph -- one sigmoid feeds both the value and the gradient

d_sigmoid nodes  before optimize: 3   after: 1


graph(%0:f64[3,4]) {
  %1 = sigmoid %0 -> f64[3,4]
  %2 = sum %1 -> f64[]
  %7 = sub 1.0 %1 -> f64[3,4]
  %8 = mul %1 %7 -> f64[3,4]
  outputs: %2, %8
}

## 4. Reverse mode *derived from* forward mode: `vjp_graph`

`vjp_graph = transpose ∘ linearize` builds the same `(value, grads)` graph a
different way: it linearizes the pipeline (reusing the forward/JVP rules) into a
graph linear in the tangents, then transposes only the linear part. Same gradient,
derived from forward mode.

In [7]:
vjp_graph(forward, x)

graph(%0:f64[4,3]) {
  %2 = matmul %0 f64[3,2] -> f64[4,2]
  %6 = add %2 f64[2] -> f64[4,2]
  %8 = tanh %6 -> f64[4,2]
  %9 = tanh %6 -> f64[4,2]
  %10 = tanh %6 -> f64[4,2]
  %11 = mul %9 %10 -> f64[4,2]
  %12 = sub 1.0 %11 -> f64[4,2]
  %14 = mul %8 %8 -> f64[4,2]
  %18 = sum %14 {axis=None, keepdims=False} -> f64[]
  %20 = const 1.0 -> f64[]
  %21 = broadcast_to %20 [4, 2] -> f64[4,2]
  %23 = mul %21 %8 -> f64[4,2]
  %25 = mul %21 %8 -> f64[4,2]
  %26 = add %23 %25 -> f64[4,2]
  %28 = mul %26 %12 -> f64[4,2]
  %29 = matmul %28 f64[2,3] -> f64[4,3]
  outputs: %18, %29
}

## 5. Watch a pass fire on the forward

`$v` (named) feeds the *same* value to both `tanh`s, so CSE can merge them; the
unused branch is dropped by DCE.

In [8]:
redundant = $ |> np.tanh($v) * np.tanh($v) |> np.sum   # two tanh of the SAME input
r = capture(redundant, xs) 
print("before:", repr(r))
print("after :", repr(optimize(r)))   # the duplicate tanh is gone
optimize(r)

before: Graph(1 in, 4 ops, 1 out)
after : Graph(1 in, 3 ops, 1 out)


graph(%0:f64[3,4]) {
  %1 = tanh %0 -> f64[3,4]
  %3 = mul %1 %1 -> f64[3,4]
  %4 = sum %3 -> f64[]
  outputs: %4
}